In [1]:
!wget https://archive.ics.uci.edu/static/public/908/realwaste.zip
!unzip realwaste.zip

--2024-12-08 13:57:29--  https://archive.ics.uci.edu/static/public/908/realwaste.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘realwaste.zip.2’

realwaste.zip.2         [                <=> ]  62.28M  18.2MB/s               ^C
Archive:  realwaste.zip
797f4fca27a3a85e4c27131cdb7d9a9a5d72c494
replace realwaste-main/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## Importing Libraries

In [1]:
# Import PyTorch
import torch
from torch import nn

# Import torchvision
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor
import torchvision.transforms as T

import matplotlib.pyplot as plt
import glob
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"

## Setting Up DataLoaders

In [2]:
import torch
import torch.nn.functional as F

class RealWasteDataset(Dataset):
    def __init__(self, imgs_path):
        self.transform = T.Compose([T.ToPILImage(),T.Resize((112, 112)),T.ToTensor()])
        self.imgs_path = imgs_path

        file_list = glob.glob(self.imgs_path + "*")
        self.data = []
        for class_path in file_list:
            class_name = class_path.split("/")[-1]
            for img_path in glob.glob(class_path + "/*.jpg"):
                self.data.append([img_path, class_name])
        self.class_map = {"Cardboard": 0, "Food Organics": 1, "Glass": 2, "Metal": 3, "Miscellaneous Trash": 4, "Paper": 5, "Plastic": 6, "Textile Trash": 7, "Vegetation": 8}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, class_name = self.data[idx]
        img = cv2.imread(img_path)
        class_id = self.class_map[class_name]

        # Convert image to tensor and transform
        img_tensor = torch.from_numpy(img)
        if self.transform:
            img_tensor = self.transform(img_tensor.permute(2, 0, 1) / 255.0)

        # One-hot encode the class_id
        one_hot_label = F.one_hot(torch.tensor(class_id), num_classes=9).float()

        return img_tensor, one_hot_label


In [3]:
dataset = RealWasteDataset("realwaste-main/RealWaste/")

train_size, validation_size = int(0.6 * len(dataset)), int(0.2 * len(dataset))   # 80% training
test_size = len(dataset) - train_size-validation_size
train_dataset, validation_dataset, test_dataset = random_split(dataset, [train_size, validation_size, test_size])

# Create DataLoaders
BATCH_SIZE = 32
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Implementing the Model

In [8]:
import torch
from torch import nn

class ConvNeuralNet(nn.Module):
    def __init__(self, num_classes):
        super(ConvNeuralNet, self).__init__()
        self.conv_layer1 = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=5)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv_layer2 = nn.Conv2d(in_channels=8, out_channels=1, kernel_size=5)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(625, 128)  # Updated flattened size
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 9)

    def forward(self, x):
        out = self.conv_layer1(x)
        out = self.relu(out)
        out = self.max_pool1(out)

        out = self.conv_layer2(out)
        out = self.relu(out)
        out = self.max_pool2(out)

        out = out.reshape(out.size(0), -1)

        out = self.fc1(out)
        out = self.relu1(out)
        out = self.dropout1(out)
        out = self.fc2(out)
        return out


# Example of initializing the network
model = ConvNeuralNet(9).to('cuda' if torch.cuda.is_available() else 'cpu')

print(model)


ConvNeuralNet(
  (conv_layer1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv_layer2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (max_pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_layer3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (conv_layer4): Conv2d(64, 1, kernel_size=(3, 3), stride=(1, 1))
  (max_pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=625, out_features=128, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=128, out_features=9, bias=True)
)


In [9]:
from torch.optim import Adam

loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.01)

In [13]:
from tqdm.auto import tqdm
NUM_CLASSES = 9
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]"):
        images, labels = images.to(device), labels.to(device)

        # Ensure labels are a flat vector of size (batch,)
        # Currently labels might be shape (batch_size, 1)
        labels = labels.squeeze()

        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        train_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Average Train Loss: {avg_train_loss:.4f}")

    # Evaluate on test set
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.inference_mode():
        for images, labels in test_dataloader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.squeeze()

            outputs = model(images)
            loss = loss_fn(outputs, labels)
            test_loss += loss.item()

            # Accuracy
            preds = outputs.argmax(dim=1)
            correct += (preds == labels.argmax(dim=1)).sum().item()
            total += labels.shape[0]

    avg_test_loss = test_loss / len(test_dataloader)
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}, Test Loss: {avg_test_loss:.4f}, Test Accuracy: {accuracy:.4f}")


Epoch 1/5 [Training]:   0%|          | 0/119 [00:00<?, ?it/s]

Epoch 1/5, Average Train Loss: 1.6728
Epoch 1/5, Test Loss: 1.6479, Test Accuracy: 0.4101


Epoch 2/5 [Training]:   0%|          | 0/119 [00:00<?, ?it/s]

Epoch 2/5, Average Train Loss: 1.5618
Epoch 2/5, Test Loss: 1.6742, Test Accuracy: 0.4090


Epoch 3/5 [Training]:   0%|          | 0/119 [00:00<?, ?it/s]

Epoch 3/5, Average Train Loss: 1.4286
Epoch 3/5, Test Loss: 1.6819, Test Accuracy: 0.4143


Epoch 4/5 [Training]:   0%|          | 0/119 [00:00<?, ?it/s]

Epoch 4/5, Average Train Loss: 1.2871
Epoch 4/5, Test Loss: 1.7555, Test Accuracy: 0.4111


Epoch 5/5 [Training]:   0%|          | 0/119 [00:00<?, ?it/s]

Epoch 5/5, Average Train Loss: 1.1249
Epoch 5/5, Test Loss: 1.8777, Test Accuracy: 0.3764
